# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR<sup>2</sup> dataset using the `mlcroissant` library. All entities in the dataset—record sets, fields, and columns—are referenced by their `@id` fields for unambiguous access.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for the FAIR^2 dataset
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('Version:', getattr(metadata, 'version', ''))
print('License:', getattr(metadata, 'license', ''))

## 2. Data Overview
Review available record sets and fields by their `@id`, name, and description (where available).

In [ ]:
# List all record sets and their fields
for record_set in dataset.record_sets:
    print(f"Record Set @id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', None)}")
    print(f"  Description: {getattr(record_set, 'description', None)}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}, name: {getattr(field, 'name', None)}, data_type: {getattr(field, 'data_type', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field selections are referenced by their `@id`.

In [ ]:
# For this dataset, let's identify a main record set. From schema, most likely candidates are those with 'protein' or 'analysis' in the id or name.
# List all record set @ids to select from (see above cell's output if running interactively).
record_set_ids = [rs.id for rs in dataset.record_sets]
print('All record set @ids in the dataset:')
for rec_id in record_set_ids:
    print(' ', rec_id)

# ----
# Choose primary record set (by inspection or by typical main table naming convention, update if needed)
# This dataset contains one main table; below, it's likely 'cr:RecordSet/proteins' or similar (update id as per real output in section 2)
# ----
main_record_set_id = record_set_ids[0]  # Use your inspected @id, here we take the first found
print(f'Using record set: {main_record_set_id}')

# Load all records of each record set into DataFrames
dataframes = {}
for rs_id in record_set_ids:
    print(f'Reading records for: {rs_id}')
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns for the main table
main_df = dataframes[main_record_set_id]
print('\nMain DataFrame columns:')
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's explore numeric fields. We'll select a numeric field (for instance, abundance or molecular weight), filter by a high value, and examine its normalized form. Select field and group by column `@id` as shown in previous overviews.

In [ ]:
# Inspect columns to select a numeric field for EDA
print('Main DataFrame numeric candidates:')
print(main_df.select_dtypes(include='number').columns.tolist())

# Example: Let's choose 'MW (kDa)' (molecular weight) if it exists, otherwise pick a likely numeric column
numeric_field_id = None
for col in main_df.columns:
    if 'mw' in col.lower() or 'weight' in col.lower() or 'abund' in col.lower() or 'count' in col.lower():
        numeric_field_id = col
        break

if numeric_field_id is None:
    # fallback to the first numeric column
    numeric_candidates = main_df.select_dtypes(include='number').columns.tolist()
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None

print(f'Using numeric field: {numeric_field_id}')

# Filter DataFrame by high (e.g., top quartile) values
if numeric_field_id is not None:
    threshold = main_df[numeric_field_id].quantile(0.75)
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold} (75th percentile):")
    print(filtered_df.head())

    # Normalize the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a possible categorical field, like 'Sample' or 'Protein class'
    group_field_id = None
    for col in main_df.select_dtypes(exclude='number').columns:
        if 'sample' in col.lower() or 'class' in col.lower() or 'group' in col.lower():
            group_field_id = col
            break
    # Fallback: just use the first object column
    if group_field_id is None and len(main_df.select_dtypes(exclude='number').columns) > 0:
        group_field_id = main_df.select_dtypes(exclude='number').columns[0]

    if group_field_id is not None:
        print(f'Grouping by field: {group_field_id}')
        # group by and aggregate mean of normalized values
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id, f"{numeric_field_id}_normalized"].mean()
        print(grouped_df.head())
else:
    print('No numeric fields were found for EDA.')

## 5. Visualization
Visualize the distribution of the selected numeric field, both before and after normalization, and their relation to the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field's distribution (if field is set)
if numeric_field_id is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field_id}")

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True, ax=axes[1])
        axes[1].set_title(f"Normalized Distribution: {numeric_field_id}")

    plt.tight_layout()
    plt.show()

    # Optional: Boxplot by group field
    if group_field_id is not None:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=filtered_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print('No numeric field to plot.')

## 6. Conclusion
In this notebook, we loaded the FAIR<sup>2</sup> extracellular vesicle mass spectrometry dataset using `mlcroissant`, identified its structure via record set and field `@id`, and demonstrated flexible analysis:

- Listing dataset structure and referencing entities by `@id`
- Loading record sets into DataFrames
- Filtering and normalizing numeric features
- Grouping and visualizing key protein metrics by biologically meaningful groupings

**Next steps** may include detailed domain-specific analysis, feature selection, and building ML models or further statistical exploration of the protein abundance data.